# 01 — Profiling CPU

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :
- mesurer le temps CPU de votre code avec `cProfile` ;
- identifier les fonctions les plus coûteuses avec `pstats` ;
- profiler ligne par ligne avec `line_profiler` ;
- utiliser `py-spy` pour profiler un processus Python en cours d'exécution ;
- interpréter un flame graph et un profil statistique.

## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :
- les fonctions, classes, décorateurs ;
- les modules de la bibliothèque standard ;
- la ligne de commande Python (`python -m`) ;
- les bases de l'algorithmique (complexité O(n), O(n²), etc.).

## Plan

1. Pourquoi profiler ?
2. `cProfile` — le profiler intégré
3. `pstats` — analyser les résultats
4. Profiler un bloc de code avec `cProfile.Profile()`
5. `line_profiler` — profiling ligne par ligne
6. `py-spy` — profiling en production
7. Bonnes pratiques
8. Synthèse
9. Exercices
10. Ressources

---

## 1. Pourquoi profiler ?

**Premature optimization is the root of all evil** — Donald Knuth.

Le profiling répond à une question simple : **où mon programme passe-t-il son temps ?** Sans mesure, on optimise à l'aveugle.

| Approche | Problème |
|---|---|
| « Je devine » | Intuition souvent fausse (effet 80/20) |
| `time.time()` partout | Invasif, imprécis, pas maintenable |
| **Profiler** | Mesure objective, granulaire, reproductible |

---

## 2. `cProfile` — le profiler intégré

`cProfile` est un profiler **déterministe** : il enregistre chaque appel de fonction, sa durée, et le nombre d'appels. Il fait partie de la bibliothèque standard.

In [ ]:
import cProfile

### Profiler une fonction simple

In [ ]:
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

In [ ]:
cProfile.run("fibonacci(25)")

Chaque colonne a un sens précis :

| Colonne | Signification |
|---|---|
| `ncalls` | Nombre d'appels (récursifs : `total/primitifs`) |
| `tottime` | Temps passé **dans** la fonction (hors sous-appels) |
| `percall` | `tottime / ncalls` |
| `cumtime` | Temps cumulé (incluant les sous-appels) |
| `percall` (2e) | `cumtime / ncalls` |
| `filename:lineno(function)` | Localisation de la fonction |

### Profiler depuis la ligne de commande

```bash
python -m cProfile -s cumulative mon_script.py
python -m cProfile -o profil.prof mon_script.py
```

L'option `-s` trie la sortie, `-o` sauvegarde le profil dans un fichier binaire.

---

## 3. `pstats` — analyser les résultats

Le module `pstats` permet de charger un profil sauvegardé et de l'analyser de manière interactive.

In [ ]:
import pstats
import io

In [ ]:
# Profiler et capturer les résultats
profiler = cProfile.Profile()
profiler.enable()
fibonacci(20)
profiler.disable()

# Analyser avec pstats
stats = pstats.Stats(profiler)
stats.sort_stats("cumulative")
stats.print_stats(10)  # Top 10

### Filtrer par nom de fonction

In [ ]:
stats.print_stats("fibonacci")

### Qui appelle qui ? — `print_callers` et `print_callees`

In [ ]:
stats.print_callers("fibonacci")

In [ ]:
stats.print_callees("fibonacci")

### Sauvegarder et recharger un profil

In [ ]:
import tempfile, os

with tempfile.NamedTemporaryFile(suffix=".prof", delete=False) as tmp:
    profiler.dump_stats(tmp.name)
    print(f"Profil sauvegardé : {tmp.name}")

    # Recharger
    stats2 = pstats.Stats(tmp.name)
    stats2.sort_stats("tottime").print_stats(5)
    os.unlink(tmp.name)

---

## 4. Profiler un bloc de code avec `cProfile.Profile()`

Plutôt que `cProfile.run()`, le context manager `Profile` offre un contrôle plus fin.

In [ ]:
def calcul_lourd():
    total = 0
    for i in range(100_000):
        total += i ** 2
    return total

In [ ]:
profiler = cProfile.Profile()
profiler.enable()
resultat = calcul_lourd()
profiler.disable()

stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream)
stats.sort_stats("tottime")
stats.print_stats()
print(stream.getvalue())

### Créer un context manager réutilisable

In [ ]:
from contextlib import contextmanager

@contextmanager
def profiler_contexte(tri="cumulative", lignes=15):
    prof = cProfile.Profile()
    prof.enable()
    yield prof
    prof.disable()
    stats = pstats.Stats(prof)
    stats.sort_stats(tri)
    stats.print_stats(lignes)

In [ ]:
with profiler_contexte():
    fibonacci(20)

---

## 5. `line_profiler` — profiling ligne par ligne

`cProfile` mesure au niveau des **fonctions**. `line_profiler` mesure au niveau des **lignes** : il montre combien de temps chaque ligne individuelle prend.

> **Installation :** `pip install line_profiler`

### Utilisation en script

```python
# script_a_profiler.py
@profile  # décorateur magique ajouté par line_profiler
def traiter_donnees(n):
    data = list(range(n))
    carres = [x ** 2 for x in data]
    filtre = [x for x in carres if x % 3 == 0]
    return sum(filtre)

traiter_donnees(100_000)
```

```bash
kernprof -l -v script_a_profiler.py
```

### Utilisation en notebook

In [ ]:
# En notebook, on utilise line_profiler via son API
try:
    from line_profiler import LineProfiler

    def traiter_donnees(n):
        data = list(range(n))
        carres = [x ** 2 for x in data]
        filtre = [x for x in carres if x % 3 == 0]
        return sum(filtre)

    lp = LineProfiler()
    lp.add_function(traiter_donnees)
    lp.enable()
    traiter_donnees(100_000)
    lp.disable()
    lp.print_stats()
except ImportError:
    print("line_profiler non installé — pip install line_profiler")

Chaque ligne affiche :

| Colonne | Signification |
|---|---|
| `Line #` | Numéro de ligne |
| `Hits` | Nombre de fois que la ligne a été exécutée |
| `Time` | Temps total sur cette ligne |
| `Per Hit` | Temps moyen par exécution |
| `% Time` | Pourcentage du temps total de la fonction |

---

## 6. `py-spy` — profiling en production

`py-spy` est un profiler **statistique** écrit en Rust qui s'attache à un processus Python **en cours d'exécution** sans le modifier ni le ralentir significativement.

> **Installation :** `pip install py-spy`

### Commandes principales

```bash
# Flame graph d'un script
py-spy record -o flamegraph.svg -- python mon_script.py

# S'attacher à un processus existant
py-spy record -o flamegraph.svg --pid 12345

# Vue top en temps réel
py-spy top --pid 12345

# Dump des stacks à un instant t
py-spy dump --pid 12345
```

### Lire un flame graph

Un flame graph empile les appels de fonctions de bas en haut :

- **Largeur** = temps relatif passé dans cette fonction (et ses enfants).
- **Hauteur** = profondeur de la pile d'appels.
- **Couleur** = généralement aléatoire (pas de signification).

Les plateaux larges en haut de la pile sont vos **goulots d'étranglement**.

### Profiler déterministe vs statistique

| Critère | `cProfile` (déterministe) | `py-spy` (statistique) |
|---|---|---|
| Mécanisme | Instrument chaque appel | Échantillonne la pile périodiquement |
| Surcoût | 10-30 % | < 5 % |
| Granularité | Fonction | Fonction (ou ligne avec `--function`) |
| Production-safe | Non | Oui |
| Nécessite modification du code | Oui (`cProfile.run()`) | Non |
| Profils multithreads | Limité | Natif |

---

## 7. Bonnes pratiques

1. **Profilez avant d'optimiser.** Toujours.
2. **Profilez sur des données réalistes** : un profil sur 10 éléments ne prédit pas le comportement sur 10 millions.
3. **Répétez les mesures** : le bruit système (GC, cache CPU, I/O) peut fausser un profil unique.
4. **Désactivez le mode debug** : les assertions et le logging ajoutent du bruit.
5. **Commencez par `cProfile`** pour la vue d'ensemble, puis **zoomez avec `line_profiler`** sur les fonctions chaudes.
6. **En production**, préférez `py-spy` (zero-overhead, pas de modification de code).

### Workflow typique

```
1. cProfile → identifier les fonctions coûteuses (top 5)
2. line_profiler → identifier les lignes coûteuses dans ces fonctions
3. Optimiser → modifier le code
4. Re-profiler → vérifier l'amélioration
5. Benchmarker → confirmer avec timeit / pyperf (voir notebook suivant)
```

---

## 7bis. Profiling conditionnel avec `profile` decorator

In [ ]:
import functools

def profileable(fn):
    """Décorateur qui ajoute .profile() pour profiler la fonction à la demande."""
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        return fn(*args, **kwargs)

    def run_profiled(*args, **kwargs):
        prof = cProfile.Profile()
        prof.enable()
        result = fn(*args, **kwargs)
        prof.disable()
        pstats.Stats(prof).sort_stats("cumulative").print_stats(10)
        return result

    wrapper.profile = run_profiled
    return wrapper

In [ ]:
@profileable
def tri_insertion(lst):
    result = lst.copy()
    for i in range(1, len(result)):
        key = result[i]
        j = i - 1
        while j >= 0 and result[j] > key:
            result[j + 1] = result[j]
            j -= 1
        result[j + 1] = key
    return result

# Appel normal (pas de profiling)
import random
data = random.sample(range(5000), 5000)
tri_insertion(data)
print("Appel normal terminé")

In [ ]:
# Appel profilé
tri_insertion.profile(data[:500])

---

## 8. Synthèse

| Outil | Type | Granularité | Surcoût | Production |
|---|---|---|---|---|
| `cProfile` | Déterministe | Fonction | Moyen | Non |
| `line_profiler` | Déterministe | Ligne | Élevé | Non |
| `py-spy` | Statistique | Fonction/ligne | Faible | Oui |
| `pstats` | Analyse | — | — | — |

**Règles à retenir :**
- Profilez **avant** d'optimiser — pas après, pas « quand j'aurai le temps ».
- `cProfile` donne la vue d'ensemble ; `line_profiler` zoome sur le détail.
- `py-spy` est l'outil de choix en production : zéro modification, faible impact.
- Un flame graph large en haut = goulot d'étranglement.
- Profilez sur des données **réalistes** et **répétez** les mesures.

---

## 9. Exercices

### Exercice 1 — Premier profiling *(facile)*

Profilez la fonction suivante avec `cProfile` et identifiez la fonction la plus coûteuse :

```python
def mystere(n):
    result = []
    for i in range(n):
        if est_premier(i):
            result.append(i)
    return result

def est_premier(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Profiling_cpu", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
def est_premier(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

def mystere(n):
    result = []
    for i in range(n):
        if est_premier(i):
            result.append(i)
    return result

cProfile.run("mystere(10_000)")
# est_premier est appelée 10000 fois et représente la majorité du temps
```

</details>

### Exercice 2 — Comparer deux algorithmes *(moyen)*

Profilez les deux versions de calcul de la somme des carrés des nombres premiers < N :

```python
def version_naive(n):
    return sum(x**2 for x in range(n) if est_premier(x))

def version_crible(n):
    crible = [True] * n
    crible[0] = crible[1] = False
    for i in range(2, int(n**0.5) + 1):
        if crible[i]:
            for j in range(i*i, n, i):
                crible[j] = False
    return sum(i**2 for i, v in enumerate(crible) if v)
```

Comparez `tottime` et `ncalls`. Laquelle est meilleure et pourquoi ?

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Profiling_cpu", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
def version_naive(n):
    return sum(x**2 for x in range(n) if est_premier(x))

def version_crible(n):
    crible = [True] * n
    crible[0] = crible[1] = False
    for i in range(2, int(n**0.5) + 1):
        if crible[i]:
            for j in range(i*i, n, i):
                crible[j] = False
    return sum(i**2 for i, v in enumerate(crible) if v)

print("=== Naïve ===")
cProfile.run("version_naive(50_000)")

print("=== Crible ===")
cProfile.run("version_crible(50_000)")

# Le crible est beaucoup plus rapide car il évite la vérification
# individuelle de chaque nombre. La version naïve appelle est_premier
# 50000 fois, chacune faisant une boucle interne.
```

</details>

### Exercice 3 — Context manager de profiling *(moyen)*

Écrire un context manager `Profiler` (classe ou fonction) qui :
1. Profile le bloc de code ;
2. Stocke les `Stats` dans un attribut `.stats` ;
3. Affiche automatiquement le top N en sortie de bloc.

```python
with Profiler(top=5) as p:
    fibonacci(20)
# Affiche automatiquement le top 5
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Profiling_cpu", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
class Profiler:
    def __init__(self, top: int = 10, sort: str = "cumulative"):
        self.top = top
        self.sort = sort
        self._prof = cProfile.Profile()
        self.stats = None

    def __enter__(self):
        self._prof.enable()
        return self

    def __exit__(self, *exc):
        self._prof.disable()
        self.stats = pstats.Stats(self._prof)
        self.stats.sort_stats(self.sort)
        self.stats.print_stats(self.top)
        return False

with Profiler(top=5) as p:
    fibonacci(20)
```

</details>

### Exercice 4 — Optimiser un goulot *(difficile)*

Le code suivant est volontairement lent. Profilez-le, identifiez le goulot, et proposez une version optimisée :

```python
def compter_mots(texte, mots_cibles):
    resultats = {}
    for mot in mots_cibles:
        count = 0
        for token in texte.split():
            if token.lower() == mot.lower():
                count += 1
        resultats[mot] = count
    return resultats
```

Testez avec un texte de 100 000 mots et 50 mots cibles.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Profiling_cpu", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
from collections import Counter

def compter_mots_lent(texte, mots_cibles):
    resultats = {}
    for mot in mots_cibles:
        count = 0
        for token in texte.split():
            if token.lower() == mot.lower():
                count += 1
        resultats[mot] = count
    return resultats

def compter_mots_rapide(texte, mots_cibles):
    # Un seul split, un seul passage
    compteur = Counter(token.lower() for token in texte.split())
    cibles_lower = {m.lower() for m in mots_cibles}
    return {mot: compteur.get(mot.lower(), 0) for mot in mots_cibles}

# Données de test
import random
vocabulaire = [f"mot{i}" for i in range(200)]
texte = " ".join(random.choices(vocabulaire, k=100_000))
cibles = random.sample(vocabulaire, 50)

# Profiling comparatif
print("=== Version lente ===")
cProfile.run("compter_mots_lent(texte, cibles)")

print("=== Version rapide ===")
cProfile.run("compter_mots_rapide(texte, cibles)")

# La version lente appelle split() 50 fois (une par mot cible).
# La version rapide ne l'appelle qu'une fois et utilise Counter.
```

</details>

---

## 10. Ressources

- [Module `cProfile` — documentation officielle](https://docs.python.org/3/library/profile.html)
- [Module `pstats`](https://docs.python.org/3/library/profile.html#the-stats-class)
- [`line_profiler` sur PyPI](https://pypi.org/project/line-profiler/)
- [`py-spy` sur GitHub](https://github.com/benfred/py-spy)
- [Flame Graphs — Brendan Gregg](https://www.brendangregg.com/flamegraphs.html)